<a href="https://colab.research.google.com/github/oselumeseagbonrofo/small-llm-experiments/blob/main/LORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LORA Finetune using FLAN-T5 (80M Parameters)

## Import Dataset

In [1]:
from datasets import load_dataset
dataset = load_dataset("knkarthick/samsum")

## Preprocess Data

In [2]:
from transformers import AutoTokenizer
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

### Set all sequences to fixed length.

Reduce longer ones and pad shorter *ones*

In [3]:
import numpy as np
from datasets import concatenate_datasets

tokenized_inputs = concatenate_datasets([dataset["train"],
 dataset["test"]]).map(lambda x: tokenizer(x["dialogue"],
 truncation=True),
 batched=True,
 remove_columns=["dialogue", "summary"])
input_lenghts = [len(x) for x in tokenized_inputs["input_ids"]]
max_source_length = int(np.percentile(input_lenghts, 85))

tokenized_targets = concatenate_datasets([dataset["train"],
 dataset["test"]]).map(lambda x: tokenizer(x["summary"],
 truncation=True),
 batched=True,
 remove_columns=["dialogue", "summary"])
target_lenghts = [len(x) for x in tokenized_targets["input_ids"]]
max_target_length = int(np.percentile(target_lenghts, 90))

Map:   0%|          | 0/15550 [00:00<?, ? examples/s]

### Tokenize input and target sequences

In [4]:
def preprocess_function(sample,padding="max_length"):
 inputs = ["summarize: " + item for item in sample["dialogue"]]

 model_inputs = tokenizer(inputs,
 max_length=max_source_length,
 padding=padding,
 truncation=True)

 labels = tokenizer(text_target=sample["summary"],
 max_length=max_target_length,
 padding=padding,
 truncation=True)

 if padding == "max_length":
  labels["input_ids"] = [
  [(l if l != tokenizer.pad_token_id else -100)
  for l in label] for label in labels["input_ids"]
 ]

 model_inputs["labels"] = labels["input_ids"]
 return model_inputs

tokenized_dataset = dataset.map(preprocess_function,
 batched=True,
 remove_columns=["dialogue", "summary", "id"])

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

### Save preprocessed training and test datasets to disk

In [5]:
tokenized_dataset["train"].save_to_disk("data/train")
tokenized_dataset["test"].save_to_disk("data/eval")

Saving the dataset (0/1 shards):   0%|          | 0/14731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

## LORA Model Training

### Load Pretrained model

In [6]:
# Install bitsandbytes for 8-bit quantization
!pip install -qqq bitsandbytes

In [7]:
# Import the necessary libraries for model loading and 8-bit quantization
from transformers import AutoModelForSeq2SeqLM

# Load the model, which `prepare_model_for_kbit_training` will then optimize for training.
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [8]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### LoRA config

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
lora_config = LoraConfig(
 r=16,
 lora_alpha=32,
 target_modules=["q", "v"],
 lora_dropout=0.05,
 bias="none",
 task_type=TaskType.SEQ_2_SEQ_LM
)

In [11]:
!pip install --upgrade torchao
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### Create data collator

In [12]:
from transformers import DataCollatorForSeq2Seq
label_pad_token_id = -100
data_collator = DataCollatorForSeq2Seq(
 tokenizer,
 model=model,
 label_pad_token_id=label_pad_token_id,
 pad_to_multiple_of=8
)

### Set training arguments

In [13]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
output_dir="lora-flan-t5-small"
training_args = Seq2SeqTrainingArguments(
 output_dir=output_dir,
 auto_find_batch_size=True,
 learning_rate=1e-3,
 num_train_epochs=3,
 logging_dir=f"{output_dir}/logs",
 logging_strategy="steps",
 logging_steps=500,
 save_strategy="no",
 report_to="tensorboard",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Create and Run Trainer instance

In [14]:
trainer = Seq2SeqTrainer(
 model=model,
 args=training_args,
 data_collator=data_collator,
 train_dataset=tokenized_dataset["train"],
)

trainer.train()

Step,Training Loss
500,1.928370
1000,1.899681
1500,1.870161
2000,1.858294
2500,1.845221
3000,1.828025
3500,1.834219
4000,1.811110
4500,1.777471
5000,1.815340


TrainOutput(global_step=5526, training_loss=1.8393981662989094, metrics={'train_runtime': 932.8486, 'train_samples_per_second': 47.374, 'train_steps_per_second': 5.924, 'total_flos': 4154239974113280.0, 'train_loss': 1.8393981662989094, 'epoch': 3.0})

### Save trained checkpoint to disk

In [15]:
peft_model_id="results"
trainer.model.save_pretrained(peft_model_id)
tokenizer.save_pretrained(peft_model_id)

('results/tokenizer_config.json', 'results/tokenizer.json')

## Example Inference

In [17]:
import torch
from peft import PeftModel, PeftConfig
peft_model_id = "results"
config = PeftConfig.from_pretrained(peft_model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(config.base_model_name_or_path, device_map={"":0})
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(model, peft_model_id, device_map={"":0})
model.eval()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 512)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 512)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=512, out_features=384, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=512, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=384, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

Summarize tasks on the test data

In [20]:
from random import randrange
dataset = load_dataset("knkarthick/samsum")
sample = dataset['test'][randrange(len(dataset["test"]))]

input_ids = tokenizer(sample["dialogue"], return_tensors="pt", truncation=True).input_ids.cuda()
outputs = model.generate(input_ids=input_ids, max_new_tokens=10,
                         do_sample=True, top_p=0.9)
print(f"input sentence: {sample['dialogue']}\n{'---'* 20}")
print(f"summary:\n{tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]}")

input sentence: Lincoln: Heeyyy ;* whats up
Fatima: I talked to Jenson, he’s not too happy ;p
Lincoln: the place sucks??
Fatima: No, the place is ok, I think, we can go there, it’s about Alene
Lincoln: typical, dont worry about it
Fatima: He thinks she may have a depression :[
Lincoln: nothin new, everyone has it, she needs a doctor then
Fatima: But she won’t go ;/
Lincoln: so she’s destroying her life fuck it its not your problem
Fatima: It is, they’re both my friends!
Lincoln: you better think what to do if they break up
Fatima: Ehh yes Ill have a problem ;//
Lincoln: both blaming each other and talking with you about it, perfect
Fatima: Alene is just troubled… She’d been through a lot…
Lincoln: everyone has their problems, the question is are ya doin sth about them
Fatima: She has problems facing it, don’t be surprised :[
Lincoln: then it is her problem
Fatima: You are so cruel at times… o.O
Lincoln: maybe, for me its just a common sense
Fatima: Why can’t everyone be just happy???
L